# 🇳🇵 Nepali Dataset Auditor — 100% FREE (Google Gemini)

> **कुनै पैसा चाहिँदैन!** Google Gemini API पूर्णतः निःशुल्क छ।

### यो notebook ले के गर्छ:
- ✅ प्रत्येक sample को गुणस्तर जाँच गर्छ
- 🔧 गल्ती भए सुधार गर्छ
- 🗑️ Duplicate र context-mismatch हटाउँछ
- 🔄 Instruction diversity improve गर्छ
- 🧠 Tricky Q&A थप्छ
- 📤 Clean JSON फर्काउँछ

---
### 🔑 FREE API Key कसरी पाउने? (2 मिनेट)
1. [aistudio.google.com](https://aistudio.google.com) मा जानुहोस्
2. Google account बाट login गर्नुहोस्
3. **"Get API Key"** → **"Create API Key"** click गर्नुहोस्
4. Copy गरेर तल Cell 1 मा paste गर्नुहोस्

✅ No credit card. No payment. Completely free!

In [ ]:


import google.generativeai as genai
import json
import time
import re

GEMINI_API_KEY = "xxxxx"  

genai.configure(api_key=GEMINI_API_KEY)

# Use Gemini 2.0 Flash — fastest & most generous free tier
model = genai.GenerativeModel("gemini-2.0-flash")

print("✅ Gemini model ready!")
print("Model: gemini-2.0-flash (FREE)")

In [ ]:

AUDIT_SYSTEM_PROMPT = """
तपाईं एक Nepali NLP dataset quality auditor हुनुहुन्छ।
दिइएको JSON dataset का प्रत्येक sample को गुणस्तर जाँच गर्नुहोस्, गल्ती भए सुधार गर्नुहोस्।

### 🎯 मुख्य नियमहरू (STRICT):

1. CONTEXT FAITHFULNESS:
   - उत्तर केवल सन्दर्भ (context/input) बाट हुनुपर्छ
   - बाहिरको कुनै जानकारी थप्न पाइँदैन

2. NO SUMMARIZATION:
   - सन्दर्भमा भएका महत्वपूर्ण सबै बुँदा समावेश हुनुपर्छ
   - उत्तर छोट्याउन पाइँदैन यदि धेरै जानकारी छ भने

3. NO HALLUCINATION:
   - अनुमान, व्याख्या, वा अतिरिक्त जानकारी निषेध

4. COMPLETE ANSWER:
   - यदि सन्दर्भमा time + condition + action छ भने → सबै समावेश गर्नुपर्छ

5. VARIATION IMPROVEMENT:
   - समान प्रकारका प्रश्नहरूलाई फरक शैलीमा rewrite गर्नुहोस्:
     - conversational (जस्तै: "मलाई भन्नुस्...")
     - indirect (जस्तै: "के तपाईं बताउन सक्नुहुन्छ...")
     - implicit (जस्तै: "...को बारेमा के थाहा छ?")
   - केवल सानो paraphrase मात्र नगर्नुहोस्

6. INSTRUCTION DIVERSITY:
   - एउटै instruction दोहोरिन नदिनुहोस्
   - फरक शैलीमा rewrite गर्नुहोस् (same meaning, different form)

7. KEEP LANGUAGE:
   - सबै सामग्री Nepali मा राख्नुहोस्

### 🔄 सुधार गर्दा:
   - अर्थ परिवर्तन नगर्नुहोस्
   - सन्दर्भ नबदल्नुहोस्
   - केवल instruction / output सुधार्नुहोस्

### ⚠️ हटाउने नियम:
   यदि sample:
   - पूर्ण duplicate छ → हटाउनुहोस्
   - context सँग output मेल खाँदैन → हटाउनुहोस्

### 📤 OUTPUT FORMAT — VERY IMPORTANT:
❗ केवल raw JSON array फर्काउनुहोस्
❗ कुनै explanation, heading, text नलेख्नुहोस्
❗ कुनै markdown (```json) नलेख्नुहोस्
❗ पहिलो character [ हुनुपर्छ, अन्तिम ] हुनुपर्छ

[
  {
    "instruction": "...",
    "input": "...",
    "output": "..."
  }
]
"""

TRICKY_SYSTEM_PROMPT = """
तपाईं एक Nepali NLP dataset creator हुनुहुन्छ।
दिइएको context बाट TRICKY (चलाख/कठिन) question-answer pairs बनाउनुहोस्।

TRICKY प्रश्नका प्रकारहरू:
- Negative questions: "के X हुँदैन?", "X नभएको अवस्थामा..."
- Conditional: "यदि Y भए, के हुन्थ्यो?"
- Multi-part: एकैसाथ दुई-तीन कुरा सोध्ने
- Implicit logic: सिधा नसोधी अर्थ बुझ्नुपर्ने
- Comparison: "X र Y मा के फरक छ?"
- Why/How reasoning: "किन", "कसरी" प्रश्न

नियमहरू:
- Context बाट मात्र उत्तर — hallucination निषेध
- Complete answer — सबै relevant points समावेश
- Nepali मा मात्र

❗ केवल raw JSON array — no markdown, no explanation:
[
  {
    "instruction": "...",
    "input": "...",
    "output": "..."
  }
]
"""

print("✅ Prompts loaded!")

In [ ]:


# 📝 Option A: Direct paste
RAW_DATASET = [
    {
        "instruction": "के हो",
        "input": "नेपाल एक सुन्दर देश हो। यहाँ हिमाल, पहाड र तराई छन्। राजधानी काठमाडौं हो। जनसंख्या करिब ३ करोड छ।",
        "output": "नेपाल"
    },
    {
        "instruction": "नेपालको राजधानी कहाँ छ?",
        "input": "नेपाल एक सुन्दर देश हो। यहाँ हिमाल, पहाड र तराई छन्। राजधानी काठमाडौं हो। जनसंख्या करिब ३ करोड छ।",
        "output": "काठमाडौं"
    },
    {
        "instruction": "नेपालको राजधानी कहाँ छ?",
        "input": "नेपाल एक सुन्दर देश हो। यहाँ हिमाल, पहाड र तराई छन्। राजधानी काठमाडौं हो। जनसंख्या करिब ३ करोड छ।",
        "output": "काठमाडौं"
    },
    {
        "instruction": "नेपालमा कति मानिस बस्छन्?",
        "input": "नेपाल एक सुन्दर देश हो। यहाँ हिमाल, पहाड र तराई छन्। राजधानी काठमाडौं हो। जनसंख्या करिब ३ करोड छ।",
        "output": "करिब ३ करोड"
    },
    {
        "instruction": "नेपालमा के के भौगोलिक क्षेत्र छन्?",
        "input": "नेपाल एक सुन्दर देश हो। यहाँ हिमाल, पहाड र तराई छन्। राजधानी काठमाडौं हो। जनसंख्या करिब ३ करोड छ।",
        "output": "हिमाल, पहाड र तराई"
    },
    {
        "instruction": "नेपालको जनसंख्या कति छ?",
        "input": "नेपाल एक सुन्दर देश हो। यहाँ हिमाल, पहाड र तराई छन्। राजधानी काठमाडौं हो। जनसंख्या करिब ३ करोड छ।",
        "output": "नेपालको जनसंख्या ५ करोड भन्दा बढी छ।"  # wrong answer — should be caught
    }
]


print(f"✅ Dataset loaded! Total: {len(RAW_DATASET)} samples")
print("\nPreview (first sample):")
print(json.dumps(RAW_DATASET[0], ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# CELL 4: Helper functions
# ============================================================

def clean_json_response(text):
    """Extract clean JSON from model response"""
    text = text.strip()
    # Remove markdown code blocks
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    text = text.strip()
    # Find JSON array
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        return match.group(0)
    return text


def call_gemini(system_prompt, user_message, retries=3):
    """Call Gemini API with retry on rate limit"""
    full_prompt = system_prompt + "\n\nUSER INPUT:\n" + user_message
    
    for attempt in range(retries):
        try:
            response = model.generate_content(full_prompt)
            return response.text
        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "quota" in error_str.lower():
                wait_time = 30 * (attempt + 1)
                print(f"  ⏳ Rate limit hit. Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"  ❌ API error: {e}")
                return None
    return None


def audit_batch(batch, batch_num, total):
    """Audit one batch of samples"""
    batch_json = json.dumps(batch, ensure_ascii=False, indent=2)
    user_msg = f"यो Nepali dataset audit र fix गर्नुहोस्:\n\n{batch_json}"
    
    print(f"  ⏳ Batch {batch_num}/{total} ({len(batch)} samples)...", end="")
    
    raw = call_gemini(AUDIT_SYSTEM_PROMPT, user_msg)
    if not raw:
        print(" ❌ Failed, keeping original")
        return batch
    
    try:
        cleaned_json = clean_json_response(raw)
        result = json.loads(cleaned_json)
        print(f" ✅ {len(batch)}→{len(result)} samples")
        return result
    except json.JSONDecodeError:
        print(f" ⚠️ Parse error, keeping original")
        return batch


print("✅ Helper functions ready!")

In [ ]:
# ============================================================
# CELL 5: 🚀 Run the Audit!
# ============================================================

BATCH_SIZE = 5  # 5 samples at a time (adjust if needed)
DELAY_BETWEEN_BATCHES = 2  # seconds between API calls

print("="*60)
print(f"🚀 STARTING AUDIT")
print(f"   Total samples : {len(RAW_DATASET)}")
print(f"   Batch size    : {BATCH_SIZE}")
print(f"   Total batches : {(len(RAW_DATASET) + BATCH_SIZE - 1) // BATCH_SIZE}")
print("="*60)

batches = [RAW_DATASET[i:i+BATCH_SIZE] for i in range(0, len(RAW_DATASET), BATCH_SIZE)]
CLEANED_DATASET = []

for i, batch in enumerate(batches, 1):
    result = audit_batch(batch, i, len(batches))
    CLEANED_DATASET.extend(result)
    if i < len(batches):
        time.sleep(DELAY_BETWEEN_BATCHES)

print("="*60)
print(f"\n✅ AUDIT DONE!")
print(f"   Before : {len(RAW_DATASET)} samples")
print(f"   After  : {len(CLEANED_DATASET)} samples")
if len(RAW_DATASET) > len(CLEANED_DATASET):
    print(f"   Removed: {len(RAW_DATASET) - len(CLEANED_DATASET)} (duplicates/mismatches)")

print("\nCleaned samples preview:")
for s in CLEANED_DATASET[:2]:
    print(json.dumps(s, ensure_ascii=False, indent=2))
    print()

In [ ]:
# ============================================================
# CELL 6: 🧠 Generate TRICKY Q&A samples
# ============================================================

TRICKY_PER_CONTEXT = 3   # each context बाट कति tricky samples?
MAX_CONTEXTS = 3          # कति contexts process गर्ने?

# Get unique contexts
seen = set()
unique_contexts = []
for s in CLEANED_DATASET:
    ctx = s.get("input", "").strip()
    if ctx and ctx not in seen:
        seen.add(ctx)
        unique_contexts.append(ctx)

print(f"Found {len(unique_contexts)} unique contexts")
print(f"Generating {TRICKY_PER_CONTEXT} tricky samples each for {min(MAX_CONTEXTS, len(unique_contexts))} contexts...\n")

TRICKY_SAMPLES = []

for ctx in unique_contexts[:MAX_CONTEXTS]:
    print(f"Context: '{ctx[:70]}...'")
    
    user_msg = f"""{ctx} — यो context बाट {TRICKY_PER_CONTEXT} वटा tricky question-answer pairs बनाउनुहोस्।

Context: {ctx}"""
    
    raw = call_gemini(TRICKY_SYSTEM_PROMPT, user_msg)
    if raw:
        try:
            parsed = json.loads(clean_json_response(raw))
            TRICKY_SAMPLES.extend(parsed)
            print(f"  ✅ Generated {len(parsed)} tricky samples")
        except json.JSONDecodeError:
            print(f"  ⚠️ Could not parse response")
    time.sleep(2)

print(f"\n🎯 Total tricky samples: {len(TRICKY_SAMPLES)}")
if TRICKY_SAMPLES:
    print("\nExample tricky sample:")
    print(json.dumps(TRICKY_SAMPLES[0], ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# CELL 7: 💾 Save & Download Final Dataset
# ============================================================

FINAL_DATASET = CLEANED_DATASET + TRICKY_SAMPLES

print("📦 FINAL DATASET SUMMARY")
print("-"*40)
print(f"  Original samples : {len(RAW_DATASET)}")
print(f"  Cleaned samples  : {len(CLEANED_DATASET)}")
print(f"  Tricky samples   : {len(TRICKY_SAMPLES)}")
print(f"  TOTAL FINAL      : {len(FINAL_DATASET)}")
print("-"*40)

# Save
OUTPUT_FILE = "nepali_dataset_cleaned_FINAL.json"
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(FINAL_DATASET, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved: {OUTPUT_FILE}")

# Download
from google.colab import files
files.download(OUTPUT_FILE)
print("📥 Download started!")

# Print full final dataset
print("\n" + "="*60)
print("COMPLETE FINAL DATASET:")
print("="*60)
print(json.dumps(FINAL_DATASET, ensure_ascii=False, indent=2))

In [ ]:
print("Uncomment above to save directly to Google Drive.")